# FINDETECT — Autoencoder training and selection

This notebook documents the training run used to select and freeze the FINDETECT autoencoder across 64 data buckets. The **final configuration uses 600,000 Train transactions and seed 123**. Increasing the Train sample to 1.2 million reduced Validation PR-AUC by approximately **17.18%** and Top-5% recall by **5.58 percentage points**, so the 600,000-row configuration was retained.

The selected model is a `24 → 12 → 6 → 12 → 24` autoencoder. Its `mean_all` anomaly score is calibrated against Train reference distributions. `Is Laundering` is used only for evaluation; `Pattern Type` is never loaded. Model selection uses Validation, and the chosen model is confirmed once on Test. The saved threshold is diagnostic, not the final operational alert threshold.

**Inputs:** the prepared numerical transaction buckets and Behavioral model scores. **Outputs:** the selected model, preprocessing and scoring package, and reports. Input and output locations are set in the configuration cell. Run cells in order to reproduce the original selection process; the saved configuration is the decision above.

## Data and reproducible sampling

Set the data contract, verify source files and schemas, and draw fixed samples by bucket and week. Test remains unopened.

In [ ]:
# 1. Runtime setup
import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "duckdb==1.4.1", "pyarrow", "pandas", "numpy",
    "scikit-learn", "joblib"
])

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; Drive mount skipped.")

from pathlib import Path
from datetime import datetime, timezone
import gc, hashlib, json, math, re, shutil

import duckdb
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix,
)

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("DuckDB:", duckdb.__version__)


In [ ]:
# 2. Configuration
DATA_DIR = Path(
    "/content/drive/MyDrive/processed/FINDETECT_Block_D_Numerical_Final"
)
BEHAVIORAL_SCORE_DIR = Path(
    "/content/drive/MyDrive/processed/FINDETECT_Behavioral_Model/"
    "full_expanding_4be2dce3cdb8/behavioral_scores"
)
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/processed/FINDETECT_Autoencoder_Model"
)

BUCKET_IDS = list(range(64))
TRAIN_START = "2022-08-08 00:00:00"
TRAIN_END   = "2022-10-03 00:00:00"
VAL_END     = "2022-10-17 00:00:00"
TEST_END    = "2022-10-30 00:00:00"
EXPECTED_MODELING_ROWS = 150_313_983

TRAIN_SIZES = [100_000, 300_000, 600_000]
VALIDATION_SAMPLE_SIZE = 300_000
TEST_SAMPLE_SIZE = 300_000

EPOCHS = 100
BATCH_SIZE = 1024
PREDICT_BATCH_SIZE = 4096
QUANTILE_SUBSAMPLE = 200_000
CORRELATION_THRESHOLD = 0.995
VARIANCE_FLOOR = 1e-8
ALERT_PERCENTILES = [90, 92, 95, 97, 98, 99, 99.5]
FORCE_REBUILD_CANDIDATES = False

# Frozen internal gate from V3; this is a project gate, not a regulatory standard.
MIN_ROC_AUC = 0.65
MIN_PR_LIFT = 2.0
MIN_TOP5_RECALL = 0.10

ORIGINAL_RAW_FEATURES = [
    "Amount Received", "Amount Paid",
    "sender_out_degree", "sender_in_degree",
    "sender_unique_counterparties",
    "receiver_out_degree", "receiver_in_degree",
    "receiver_unique_counterparties",
    "sender_receiver_frequency", "bidirectional_relationship",
    "sender_outgoing_count_so_far_wk",
    "sender_unique_receivers_so_far_wk",
    "outgoing_count_24h", "outgoing_sum_24h",
    "outgoing_count_72h", "outgoing_sum_72h",
    "sender_total_sent_prev_week", "sender_avg_sent_prev_week",
    "sender_max_sent_prev_week", "sender_outgoing_count_prev_week",
    "receiver_total_received_prev_week",
    "receiver_avg_received_prev_week",
    "receiver_max_received_prev_week",
    "receiver_incoming_count_prev_week",
    "amount_vs_prev_max_diff", "amount_vs_prev_max_ratio",
    "transaction_frequency_diff", "transaction_frequency_ratio",
    "hours_since_last_incoming", "outgoing_to_incoming_ratio",
]

# Frozen V3 feature policy.
MODEL_RAW_FEATURES = [x for x in ORIGINAL_RAW_FEATURES if x != "Amount Received"]
BINARY_SOURCE_FEATURES = ["bidirectional_relationship"]
CONTINUOUS_FEATURES = [x for x in MODEL_RAW_FEATURES if x not in BINARY_SOURCE_FEATURES]
DERIVED_BINARY_FEATURES = ["no_previous_incoming", "undefined_history_ratio"]
PRE_CLEAN_MODEL_FEATURES = CONTINUOUS_FEATURES + BINARY_SOURCE_FEATURES + DERIVED_BINARY_FEATURES
SELECTED_AGGREGATION_METHOD = "mean_all"

signature = hashlib.sha256(json.dumps({
    "buckets": BUCKET_IDS,
    "dates": [TRAIN_START, TRAIN_END, VAL_END, TEST_END],
    "train_sizes": TRAIN_SIZES,
    "validation_sample": VALIDATION_SAMPLE_SIZE,
    "test_sample": TEST_SAMPLE_SIZE,
    "features": PRE_CLEAN_MODEL_FEATURES,
    "architecture": "n-24-12-6-12-24-n",
    "aggregation": SELECTED_AGGREGATION_METHOD,
    "stratification": "bucket_x_week_nested",
    "seed": SEED,
}, sort_keys=True).encode()).hexdigest()[:12]

RUN_DIR = OUTPUT_ROOT / f"final64_learning_curve_{signature}"
CANDIDATE_DIR = RUN_DIR / "candidates"
FINAL_MODEL_DIR = RUN_DIR / "final_model"
REPORT_DIR = RUN_DIR / "reports"
for p in [CANDIDATE_DIR, FINAL_MODEL_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Run directory:", RUN_DIR)
print("Train candidates:", TRAIN_SIZES)
print("Aggregation method frozen from V3:", SELECTED_AGGREGATION_METHOD)


In [ ]:
# 3. Resolve all 64 D and Behavioral files and validate schemas
def qid(name):
    return '"' + str(name).replace('"', '""') + '"'

def sql_lit(value):
    return "'" + str(value).replace("'", "''") + "'"

def d_file(bucket_id):
    return DATA_DIR / f"block_d_numerical_bucket_{bucket_id:03d}.parquet"

def behavioral_file(bucket_id):
    return BEHAVIORAL_SCORE_DIR / f"behavioral_score_bucket_{bucket_id:03d}.parquet"

d_missing = [str(d_file(b)) for b in BUCKET_IDS if not d_file(b).exists()]
behavior_missing = [str(behavioral_file(b)) for b in BUCKET_IDS if not behavioral_file(b).exists()]
if d_missing:
    raise FileNotFoundError("Missing D files: " + repr(d_missing))
if behavior_missing:
    raise FileNotFoundError("Missing Behavioral files: " + repr(behavior_missing))

con = duckdb.connect()
con.execute("PRAGMA threads=4")
con.execute("PRAGMA memory_limit='10GB'")
DUCK_TMP = Path("/content/duckdb_autoencoder_final64_tmp")
DUCK_TMP.mkdir(parents=True, exist_ok=True)
con.execute(f"PRAGMA temp_directory={sql_lit(DUCK_TMP)}")

required_d_columns = set(
    ["Timestamp", "Is Laundering", "sender_id", "receiver_id"]
    + ORIGINAL_RAW_FEATURES
)
d_schemas = {}
for b in [0, 63]:
    cols = set(con.execute(
        f"DESCRIBE SELECT * FROM read_parquet({sql_lit(d_file(b))})"
    ).df().column_name)
    d_schemas[b] = cols
    missing = required_d_columns - cols
    if missing:
        raise ValueError(f"D bucket {b:03d} is missing columns: {sorted(missing)}")

key_presence = {b: ("transaction_key" in cols) for b, cols in d_schemas.items()}
if len(set(key_presence.values())) != 1:
    raise ValueError(f"Inconsistent transaction_key schema across D buckets: {key_presence}")
HAS_SOURCE_TRANSACTION_KEY = key_presence[0]
TRANSACTION_KEY_MODE = (
    "existing_d_column" if HAS_SOURCE_TRANSACTION_KEY else "bucket_file_row_number"
)

required_behavior_columns = {"transaction_key", "data_split", "behavioral_score"}
for b in [0, 63]:
    cols = set(con.execute(
        f"DESCRIBE SELECT * FROM read_parquet({sql_lit(behavioral_file(b))})"
    ).df().column_name)
    missing = required_behavior_columns - cols
    if missing:
        raise ValueError(f"Behavioral bucket {b:03d} is missing columns: {sorted(missing)}")

print("D files found:", len(BUCKET_IDS))
print("Behavioral files found:", len(BUCKET_IDS))
print("Schema checks passed on boundary buckets 000 and 063")
print("Transaction-key mode:", TRANSACTION_KEY_MODE)


In [ ]:
# 4. Count the complete modeling population by bucket, split, and week
split_case = f"""
CASE
  WHEN {qid('Timestamp')} >= TIMESTAMP {sql_lit(TRAIN_START)}
   AND {qid('Timestamp')} <  TIMESTAMP {sql_lit(TRAIN_END)} THEN 'train'
  WHEN {qid('Timestamp')} >= TIMESTAMP {sql_lit(TRAIN_END)}
   AND {qid('Timestamp')} <  TIMESTAMP {sql_lit(VAL_END)} THEN 'validation'
  WHEN {qid('Timestamp')} >= TIMESTAMP {sql_lit(VAL_END)}
   AND {qid('Timestamp')} <  TIMESTAMP {sql_lit(TEST_END)} THEN 'test'
END
"""

count_parts = []
for b in BUCKET_IDS:
    part = con.execute(f"""
        SELECT
            {b}::INTEGER AS bucket_id,
            {split_case} AS data_split,
            CAST(date_trunc('week', {qid('Timestamp')}) AS DATE) AS week_start,
            COUNT(*)::BIGINT AS population_rows
        FROM read_parquet({sql_lit(d_file(b))})
        WHERE {qid('Timestamp')} >= TIMESTAMP {sql_lit(TRAIN_START)}
          AND {qid('Timestamp')} <  TIMESTAMP {sql_lit(TEST_END)}
        GROUP BY 1,2,3
        ORDER BY 2,3
    """).df()
    count_parts.append(part)
    if (b + 1) % 8 == 0:
        print(f"Counted {b + 1}/64 buckets")

stratum_counts = pd.concat(count_parts, ignore_index=True)
stratum_counts["week_start"] = pd.to_datetime(stratum_counts.week_start).dt.date
population_by_split = (
    stratum_counts.groupby("data_split", as_index=False).population_rows.sum()
)
modeling_rows = int(stratum_counts.population_rows.sum())

display(population_by_split)
print("Modeling-period rows:", f"{modeling_rows:,}")
if modeling_rows != EXPECTED_MODELING_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_MODELING_ROWS:,} modeling rows but found {modeling_rows:,}. "
        "Stop and verify the D files and temporal contract."
    )

stratum_counts.to_csv(REPORT_DIR / "population_by_bucket_split_week.csv", index=False)
population_by_split.to_csv(REPORT_DIR / "population_by_split.csv", index=False)


In [ ]:
# 5. Allocate exact nested Train samples and fixed Validation/Test samples
def largest_remainder(pop, target, output_col):
    out = pop.copy().reset_index(drop=True)
    exact = out.population_rows / out.population_rows.sum() * int(target)
    out[output_col] = np.floor(exact).astype(int)
    remainder = int(target - out[output_col].sum())
    order = (exact - np.floor(exact)).sort_values(ascending=False).index[:remainder]
    out.loc[order, output_col] += 1
    if int(out[output_col].sum()) != int(target):
        raise AssertionError(f"Allocation for {output_col} does not sum to {target}")
    if (out[output_col] > out.population_rows).any():
        raise AssertionError(f"Allocation for {output_col} exceeds population")
    return out

def nested_from_parent(parent, parent_col, target, output_col):
    out = parent.copy().reset_index(drop=True)
    exact = out[parent_col] / out[parent_col].sum() * int(target)
    out[output_col] = np.floor(exact).astype(int)
    remainder = int(target - out[output_col].sum())
    fractions = exact - np.floor(exact)
    candidates = out.index[out[output_col] < out[parent_col]]
    order = fractions.loc[candidates].sort_values(ascending=False).index
    for idx in order[:remainder]:
        out.loc[idx, output_col] += 1
    if int(out[output_col].sum()) != int(target):
        raise AssertionError(f"Nested allocation for {output_col} does not sum to {target}")
    if (out[output_col] > out[parent_col]).any():
        raise AssertionError(f"{output_col} is not nested inside {parent_col}")
    return out

train_pop = stratum_counts[stratum_counts.data_split == "train"].copy()
validation_pop = stratum_counts[stratum_counts.data_split == "validation"].copy()
test_pop = stratum_counts[stratum_counts.data_split == "test"].copy()

train_alloc = largest_remainder(train_pop, 600_000, "quota_600k")
train_alloc = nested_from_parent(train_alloc, "quota_600k", 300_000, "quota_300k")
train_alloc = nested_from_parent(train_alloc, "quota_300k", 100_000, "quota_100k")
validation_alloc = largest_remainder(validation_pop, VALIDATION_SAMPLE_SIZE, "sample_rows")
test_alloc = largest_remainder(test_pop, TEST_SAMPLE_SIZE, "sample_rows")

assert (train_alloc.quota_100k <= train_alloc.quota_300k).all()
assert (train_alloc.quota_300k <= train_alloc.quota_600k).all()

train_alloc.to_csv(REPORT_DIR / "train_nested_allocations.csv", index=False)
validation_alloc.to_csv(REPORT_DIR / "validation_allocations.csv", index=False)
test_alloc.to_csv(REPORT_DIR / "test_allocations_locked_before_selection.csv", index=False)

display(pd.DataFrame([
    {"sample": "train_100k", "rows": int(train_alloc.quota_100k.sum())},
    {"sample": "train_300k", "rows": int(train_alloc.quota_300k.sum())},
    {"sample": "train_600k", "rows": int(train_alloc.quota_600k.sum())},
    {"sample": "validation", "rows": int(validation_alloc.sample_rows.sum())},
    {"sample": "test_locked_not_loaded", "rows": int(test_alloc.sample_rows.sum())},
]))


In [ ]:
# 6. Load the 600K Train master and fixed Validation sample.
# Test remains unopened at this stage.
select_features_sql = ",\n".join(qid(x) for x in ORIGINAL_RAW_FEATURES)

def load_train_master():
    parts = []
    for b in BUCKET_IDS:
        transaction_key_sql = (
            f"CAST({qid('transaction_key')} AS VARCHAR)"
            if HAS_SOURCE_TRANSACTION_KEY
            else f"printf('B%03d_R%012d', {b}, file_row_number)"
        )
        q = train_alloc[train_alloc.bucket_id == b][
            ["week_start", "quota_100k", "quota_300k", "quota_600k"]
        ].copy()
        con.register("quota_current", q)
        part = con.execute(f"""
            WITH base AS (
                SELECT
                    file_row_number,
                    {transaction_key_sql} AS transaction_key,
                    {b}::INTEGER AS d_bucket_id,
                    CAST(date_trunc('week', {qid('Timestamp')}) AS DATE) AS week_start,
                    {qid('Timestamp')},
                    CAST({qid('Is Laundering')} AS TINYINT) AS evaluation_label,
                    CAST(sender_id AS VARCHAR) AS sender_id,
                    CAST(receiver_id AS VARCHAR) AS receiver_id,
                    {select_features_sql}
                FROM read_parquet({sql_lit(d_file(b))}, file_row_number=true)
                WHERE {qid('Timestamp')} >= TIMESTAMP {sql_lit(TRAIN_START)}
                  AND {qid('Timestamp')} <  TIMESTAMP {sql_lit(TRAIN_END)}
            ), ranked AS (
                SELECT base.*, q.quota_100k, q.quota_300k, q.quota_600k,
                       ROW_NUMBER() OVER (
                           PARTITION BY base.week_start
                           ORDER BY hash(file_row_number + {SEED + 1000}), file_row_number
                       ) AS sample_rank
                FROM base
                JOIN quota_current q USING (week_start)
            )
            SELECT * EXCLUDE(file_row_number)
            FROM ranked
            WHERE sample_rank <= quota_600k
        """).df()
        con.unregister("quota_current")
        parts.append(part)
        if (b + 1) % 8 == 0:
            print(f"Loaded Train master from {b + 1}/64 buckets")
    out = pd.concat(parts, ignore_index=True)
    if len(out) != 600_000 or out.transaction_key.duplicated().any():
        raise RuntimeError("Train master row/key gate failed")
    return out

def load_fixed_split_sample(split, allocation, seed_offset):
    start_ts, end_ts = {
        "validation": (TRAIN_END, VAL_END),
        "test": (VAL_END, TEST_END),
    }[split]
    parts = []
    for b in BUCKET_IDS:
        transaction_key_sql = (
            f"CAST({qid('transaction_key')} AS VARCHAR)"
            if HAS_SOURCE_TRANSACTION_KEY
            else f"printf('B%03d_R%012d', {b}, file_row_number)"
        )
        q = allocation[allocation.bucket_id == b][["week_start", "sample_rows"]].copy()
        con.register("quota_current", q)
        part = con.execute(f"""
            WITH base AS (
                SELECT
                    file_row_number,
                    {transaction_key_sql} AS transaction_key,
                    {b}::INTEGER AS d_bucket_id,
                    CAST(date_trunc('week', {qid('Timestamp')}) AS DATE) AS week_start,
                    {qid('Timestamp')},
                    CAST({qid('Is Laundering')} AS TINYINT) AS evaluation_label,
                    CAST(sender_id AS VARCHAR) AS sender_id,
                    CAST(receiver_id AS VARCHAR) AS receiver_id,
                    {select_features_sql}
                FROM read_parquet({sql_lit(d_file(b))}, file_row_number=true)
                WHERE {qid('Timestamp')} >= TIMESTAMP {sql_lit(start_ts)}
                  AND {qid('Timestamp')} <  TIMESTAMP {sql_lit(end_ts)}
            ), ranked AS (
                SELECT base.*, q.sample_rows,
                       ROW_NUMBER() OVER (
                           PARTITION BY base.week_start
                           ORDER BY hash(file_row_number + {SEED} + {seed_offset}), file_row_number
                       ) AS sample_rank
                FROM base
                JOIN quota_current q USING (week_start)
            )
            SELECT * EXCLUDE(file_row_number, sample_rows, sample_rank)
            FROM ranked
            WHERE sample_rank <= sample_rows
        """).df()
        con.unregister("quota_current")
        parts.append(part)
        if (b + 1) % 8 == 0:
            print(f"Loaded {split} from {b + 1}/64 buckets")
    out = pd.concat(parts, ignore_index=True)
    expected = VALIDATION_SAMPLE_SIZE if split == "validation" else TEST_SAMPLE_SIZE
    if len(out) != expected or out.transaction_key.duplicated().any():
        raise RuntimeError(f"{split} row/key gate failed")
    return out

train_master_df = load_train_master()
validation_df = load_fixed_split_sample("validation", validation_alloc, 2000)

print("Train master:", train_master_df.shape)
print("Validation:", validation_df.shape)
print("Test has not been loaded.")


## Preprocessing and model definition

Audit the Train sample, define the frozen preprocessing and scoring functions, and fix the active feature schema using Train data.

In [ ]:
# 7. Representativeness audit: strata, missingness, quantiles, and account diversity
train_population_rows = int(train_pop.population_rows.sum())
sample_fraction_of_train = len(train_master_df) / train_population_rows

strata_audit = train_alloc[["bucket_id", "week_start", "population_rows", "quota_600k"]].copy()
strata_audit["population_share"] = strata_audit.population_rows / strata_audit.population_rows.sum()
strata_audit["sample_share"] = strata_audit.quota_600k / strata_audit.quota_600k.sum()
strata_audit["absolute_share_difference"] = (
    strata_audit.sample_share - strata_audit.population_share
).abs()
strata_audit.to_csv(REPORT_DIR / "representativeness_bucket_week.csv", index=False)

all_d_paths_sql = "[" + ",".join(sql_lit(d_file(b)) for b in BUCKET_IDS) + "]"
audit_exprs = [
    "COUNT(*)::BIGINT AS full_train_rows",
    "approx_count_distinct(CAST(sender_id AS VARCHAR))::BIGINT AS approx_unique_senders",
    "approx_count_distinct(CAST(receiver_id AS VARCHAR))::BIGINT AS approx_unique_receivers",
]
for i, feature in enumerate(ORIGINAL_RAW_FEATURES):
    value = f"TRY_CAST({qid(feature)} AS DOUBLE)"
    valid = f"{value} IS NOT NULL AND isfinite({value})"
    audit_exprs.append(
        f"AVG(CASE WHEN {valid} THEN 0.0 ELSE 1.0 END) AS missing_{i}"
    )
    audit_exprs.append(
        f"approx_quantile({value}, [0.01,0.05,0.50,0.95,0.99]) "
        f"FILTER (WHERE {valid}) AS quantiles_{i}"
    )

full_audit_row = con.execute(f"""
    SELECT {', '.join(audit_exprs)}
    FROM read_parquet({all_d_paths_sql})
    WHERE {qid('Timestamp')} >= TIMESTAMP {sql_lit(TRAIN_START)}
      AND {qid('Timestamp')} <  TIMESTAMP {sql_lit(TRAIN_END)}
""").fetchone()

audit_columns = ["full_train_rows", "approx_unique_senders", "approx_unique_receivers"]
for i in range(len(ORIGINAL_RAW_FEATURES)):
    audit_columns += [f"missing_{i}", f"quantiles_{i}"]
full_audit = dict(zip(audit_columns, full_audit_row))

distribution_rows = []
for i, feature in enumerate(ORIGINAL_RAW_FEATURES):
    sample_values = pd.to_numeric(train_master_df[feature], errors="coerce").replace([np.inf, -np.inf], np.nan)
    sample_q = sample_values.quantile([0.01, 0.05, 0.50, 0.95, 0.99]).to_numpy()
    full_q = np.asarray(full_audit[f"quantiles_{i}"], dtype="float64")
    full_missing = float(full_audit[f"missing_{i}"])
    sample_missing = float(sample_values.isna().mean())
    scale = max(float(full_q[-1] - full_q[0]), 1e-9)
    distribution_rows.append({
        "feature": feature,
        "full_missing_rate": full_missing,
        "sample_missing_rate": sample_missing,
        "missing_rate_abs_diff": abs(sample_missing - full_missing),
        "full_q01": full_q[0], "sample_q01": sample_q[0],
        "full_q05": full_q[1], "sample_q05": sample_q[1],
        "full_q50": full_q[2], "sample_q50": sample_q[2],
        "full_q95": full_q[3], "sample_q95": sample_q[3],
        "full_q99": full_q[4], "sample_q99": sample_q[4],
        "mean_abs_quantile_error_scaled_by_q01_q99": float(np.mean(np.abs(sample_q - full_q)) / scale),
    })

distribution_audit = pd.DataFrame(distribution_rows)
distribution_audit.to_csv(REPORT_DIR / "representativeness_feature_distributions.csv", index=False)

account_audit = pd.DataFrame([{
    "full_train_rows": int(full_audit["full_train_rows"]),
    "sample_rows": len(train_master_df),
    "sample_fraction_of_train": sample_fraction_of_train,
    "full_approx_unique_senders": int(full_audit["approx_unique_senders"]),
    "sample_unique_senders": int(train_master_df.sender_id.nunique()),
    "full_approx_unique_receivers": int(full_audit["approx_unique_receivers"]),
    "sample_unique_receivers": int(train_master_df.receiver_id.nunique()),
    "max_bucket_week_share_difference": float(strata_audit.absolute_share_difference.max()),
    "max_missing_rate_difference": float(distribution_audit.missing_rate_abs_diff.max()),
}])
account_audit.to_csv(REPORT_DIR / "representativeness_summary.csv", index=False)

display(account_audit)
display(distribution_audit.sort_values(
    "mean_abs_quantile_error_scaled_by_q01_q99", ascending=False
).head(10))
print("600K / full Train =", f"{sample_fraction_of_train:.4%}")


In [ ]:
# 8. Frozen V3 preprocessing, feature cleaning, model, and scoring helpers
def build_parts(df):
    raw = df[MODEL_RAW_FEATURES].copy().replace([np.inf, -np.inf], np.nan)
    binary = pd.DataFrame(index=raw.index)
    binary["bidirectional_relationship"] = (
        raw["bidirectional_relationship"].fillna(0).clip(0, 1).astype("float32")
    )
    binary["no_previous_incoming"] = (
        raw["hours_since_last_incoming"].isna().astype("float32")
    )
    binary["undefined_history_ratio"] = (
        raw[["amount_vs_prev_max_ratio", "transaction_frequency_ratio"]]
        .isna().any(axis=1).astype("float32")
    )
    return raw[CONTINUOUS_FEATURES].copy(), binary

def fit_preprocessor(df, active_features=None):
    continuous, binary = build_parts(df)
    imputer = SimpleImputer(strategy="median")
    imputed = pd.DataFrame(
        imputer.fit_transform(continuous),
        columns=CONTINUOUS_FEATURES, index=continuous.index,
    )
    lower = imputed.quantile(0.005)
    upper = imputed.quantile(0.995)
    clipped = imputed.clip(lower=lower, upper=upper, axis="columns")
    qt = QuantileTransformer(
        n_quantiles=min(1000, len(clipped)),
        output_distribution="normal",
        subsample=min(QUANTILE_SUBSAMPLE, len(clipped)),
        random_state=SEED,
        copy=True,
    )
    scaled = pd.DataFrame(
        qt.fit_transform(clipped).astype("float32"),
        columns=CONTINUOUS_FEATURES, index=continuous.index,
    )
    preclean = pd.concat([scaled, binary], axis=1)[PRE_CLEAN_MODEL_FEATURES]
    if active_features is None:
        variances = preclean.var(axis=0)
        near_zero = variances[variances <= VARIANCE_FLOOR].index.tolist()
        candidates = [x for x in PRE_CLEAN_MODEL_FEATURES if x not in near_zero]
        corr_sample = preclean[candidates].sample(
            n=min(100_000, len(preclean)), random_state=SEED
        )
        corr = corr_sample.corr().abs()
        correlation_drops = []
        correlation_pairs = []
        for j, feature_j in enumerate(candidates):
            if feature_j in correlation_drops:
                continue
            for i in range(j):
                feature_i = candidates[i]
                if feature_i in correlation_drops:
                    continue
                value = corr.loc[feature_i, feature_j]
                if np.isfinite(value) and value >= CORRELATION_THRESHOLD:
                    correlation_drops.append(feature_j)
                    correlation_pairs.append({
                        "kept_feature": feature_i,
                        "dropped_feature": feature_j,
                        "correlation": float(value),
                    })
                    break
        active_features = [x for x in candidates if x not in correlation_drops]
    else:
        near_zero, correlation_drops, correlation_pairs = [], [], []

    package = {
        "continuous_imputer": imputer,
        "clip_lower": lower,
        "clip_upper": upper,
        "quantile_transformer": qt,
        "active_model_features": list(active_features),
    }
    X = preclean[active_features].to_numpy(dtype="float32")
    return package, X, near_zero, correlation_drops, correlation_pairs

def transform_with_package(df, package):
    continuous, binary = build_parts(df)
    imputed = pd.DataFrame(
        package["continuous_imputer"].transform(continuous),
        columns=CONTINUOUS_FEATURES, index=continuous.index,
    )
    clipped = imputed.clip(
        lower=package["clip_lower"], upper=package["clip_upper"], axis="columns"
    )
    scaled = pd.DataFrame(
        package["quantile_transformer"].transform(clipped).astype("float32"),
        columns=CONTINUOUS_FEATURES, index=continuous.index,
    )
    preclean = pd.concat([scaled, binary], axis=1)[PRE_CLEAN_MODEL_FEATURES]
    X = preclean[package["active_model_features"]].to_numpy(dtype="float32")
    if not np.isfinite(X).all():
        raise ValueError("Non-finite values remain after preprocessing")
    return X

def build_autoencoder(input_dim):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    inputs = tf.keras.Input(shape=(input_dim,), name="active_model_features")
    x = tf.keras.layers.GaussianNoise(0.01, name="small_training_noise")(inputs)
    x = tf.keras.layers.Dense(24, activation="relu", name="encoder_24")(x)
    x = tf.keras.layers.Dense(12, activation="relu", name="encoder_12")(x)
    x = tf.keras.layers.Dense(6, activation="relu", name="latent_6")(x)
    x = tf.keras.layers.Dense(12, activation="relu", name="decoder_12")(x)
    x = tf.keras.layers.Dense(24, activation="relu", name="decoder_24")(x)
    outputs = tf.keras.layers.Dense(input_dim, activation="linear", name="reconstruction")(x)
    model = tf.keras.Model(inputs, outputs, name="FINDETECT_Autoencoder_Final64")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
        loss="mse",
    )
    return model

def build_score_references(model, X_train):
    reconstructed = model.predict(X_train, batch_size=PREDICT_BATCH_SIZE, verbose=0)
    errors = np.square(X_train - reconstructed).astype("float32")
    feature_refs = [np.sort(errors[:, j]) for j in range(errors.shape[1])]
    feature_pct = np.empty_like(errors, dtype="float32")
    for j, ref in enumerate(feature_refs):
        feature_pct[:, j] = np.searchsorted(ref, errors[:, j], side="right") / len(ref)
    raw = feature_pct.mean(axis=1).astype("float32")
    transaction_ref = np.sort(raw)
    return feature_refs, transaction_ref

def score_dataframe(model, df, package):
    X = transform_with_package(df, package)
    reconstructed = model.predict(X, batch_size=PREDICT_BATCH_SIZE, verbose=0)
    errors = np.square(X - reconstructed).astype("float32")
    feature_pct = np.empty_like(errors, dtype="float32")
    for j, ref in enumerate(package["feature_error_references"]):
        feature_pct[:, j] = np.searchsorted(ref, errors[:, j], side="right") / len(ref)
    raw = feature_pct.mean(axis=1)
    ref = package["transaction_score_reference"]
    score = np.clip(np.searchsorted(ref, raw, side="right") / len(ref), 0, 1)
    return score.astype("float32")

def metrics_row(split, model_name, y, score):
    prevalence = float(y.mean())
    ap = average_precision_score(y, score) if y.sum() else np.nan
    row = {
        "split": split,
        "model": model_name,
        "rows": len(y),
        "positives": int(y.sum()),
        "prevalence": prevalence,
        "roc_auc": roc_auc_score(y, score) if len(np.unique(y)) == 2 else np.nan,
        "pr_auc_average_precision": ap,
        "pr_lift": ap / prevalence if prevalence and np.isfinite(ap) else np.nan,
    }
    for pct in [0.01, 0.05, 0.10]:
        threshold = np.quantile(score, 1 - pct)
        mask = score >= threshold
        hits = int(y[mask].sum())
        row[f"alerts_top_{int(pct*100)}pct"] = int(mask.sum())
        row[f"precision_top_{int(pct*100)}pct"] = hits / mask.sum()
        row[f"recall_top_{int(pct*100)}pct"] = hits / y.sum()
    return row


In [ ]:
# 9. Freeze the active feature schema from the 600K Train master
schema_package, _, near_zero_features, correlation_drops, correlation_pairs = (
    fit_preprocessor(train_master_df, active_features=None)
)
ACTIVE_MODEL_FEATURES = schema_package["active_model_features"]

feature_cleaning = {
    "near_zero_features": near_zero_features,
    "correlation_drops": correlation_drops,
    "active_model_features": ACTIVE_MODEL_FEATURES,
    "policy": "feature schema frozen from 600K Train only; candidate-specific scalers remain Train-subset fitted",
}
(REPORT_DIR / "feature_cleaning.json").write_text(json.dumps(feature_cleaning, indent=2))
pd.DataFrame(correlation_pairs).to_csv(REPORT_DIR / "correlation_pairs.csv", index=False)

print("Near-zero removed:", near_zero_features)
print("Correlation removals:", correlation_drops)
print("Final active model features:", len(ACTIVE_MODEL_FEATURES))
del schema_package
gc.collect()


## Validation evidence for the final decision

The cells below preserve the original comparisons needed to reproduce why 600,000 rows and seed 123 were chosen. These decisions are already final.

In [ ]:
# 10. Train 100K / 300K / 600K candidates and compare on Validation only
def train_subset(size):
    quota_col = {100_000: "quota_100k", 300_000: "quota_300k", 600_000: "quota_600k"}[size]
    return train_master_df[train_master_df.sample_rank <= train_master_df[quota_col]].copy()

y_validation = validation_df.evaluation_label.astype("int8").to_numpy()
candidate_metric_rows = []

for size in TRAIN_SIZES:
    candidate_path = CANDIDATE_DIR / f"train_{size}"
    candidate_path.mkdir(parents=True, exist_ok=True)
    model_path = candidate_path / "autoencoder.keras"
    package_path = candidate_path / "preprocessing_and_scoring.joblib"
    metrics_path = candidate_path / "validation_metrics.json"

    if (
        not FORCE_REBUILD_CANDIDATES
        and model_path.exists() and package_path.exists() and metrics_path.exists()
    ):
        row = json.loads(metrics_path.read_text())
        candidate_metric_rows.append(row)
        print(f"Train {size:,}: reused completed candidate")
        continue

    train_df = train_subset(size)
    if len(train_df) != size:
        raise RuntimeError(f"Nested Train subset expected {size:,}, found {len(train_df):,}")

    package, X_train, _, _, _ = fit_preprocessor(
        train_df, active_features=ACTIVE_MODEL_FEATURES
    )
    X_validation = transform_with_package(validation_df, package)
    model = build_autoencoder(X_train.shape[1])
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
        ),
    ]
    history = model.fit(
        X_train, X_train,
        validation_data=(X_validation, X_validation),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        callbacks=callbacks,
        verbose=1,
    )
    pd.DataFrame(history.history).to_csv(candidate_path / "training_history.csv", index=False)

    feature_refs, transaction_ref = build_score_references(model, X_train)
    package.update({
        "original_raw_features": ORIGINAL_RAW_FEATURES,
        "model_raw_features": MODEL_RAW_FEATURES,
        "continuous_features": CONTINUOUS_FEATURES,
        "preclean_model_features": PRE_CLEAN_MODEL_FEATURES,
        "feature_error_references": feature_refs,
        "transaction_score_reference": transaction_ref,
        "selected_aggregation_method": SELECTED_AGGREGATION_METHOD,
        "train_sample_size": size,
        "seed": SEED,
    })
    validation_score = score_dataframe(model, validation_df, package)
    row = metrics_row("validation", f"autoencoder_train_{size}", y_validation, validation_score)
    row["train_rows"] = size
    row["epochs_completed"] = len(history.history["loss"])
    row["best_validation_reconstruction_loss"] = float(min(history.history["val_loss"]))

    model.save(model_path)
    joblib.dump(package, package_path)
    metrics_path.write_text(json.dumps(row, indent=2))
    candidate_metric_rows.append(row)
    print(f"Train {size:,}: candidate saved")

    del train_df, X_train, X_validation, model, package, validation_score
    gc.collect()

learning_curve = pd.DataFrame(candidate_metric_rows).sort_values("train_rows").reset_index(drop=True)
learning_curve["pr_auc_gain_from_previous"] = learning_curve.pr_auc_average_precision.diff()
learning_curve["top5_recall_gain_from_previous"] = learning_curve.recall_top_5pct.diff()
learning_curve["reconstruction_loss_change_from_previous"] = (
    learning_curve.best_validation_reconstruction_loss.diff()
)
learning_curve.to_csv(REPORT_DIR / "validation_learning_curve.csv", index=False)
display(learning_curve)

# Primary selector: Validation PR-AUC; ties break by Top-5 recall, then ROC-AUC.
selected_row = learning_curve.sort_values(
    ["pr_auc_average_precision", "recall_top_5pct", "roc_auc"],
    ascending=False,
).iloc[0]
SELECTED_TRAIN_SIZE = int(selected_row.train_rows)
(REPORT_DIR / "selected_train_size.json").write_text(json.dumps({
    "selected_train_size": SELECTED_TRAIN_SIZE,
    "selection_split": "validation",
    "primary_metric": "pr_auc_average_precision",
    "tie_breakers": ["recall_top_5pct", "roc_auc"],
    "test_was_loaded_before_selection": False,
}, indent=2))

print("Validation-selected Train size:", f"{SELECTED_TRAIN_SIZE:,}")
print("Test is still unopened.")


In [ ]:
# 11. Validation comparison with Behavioral, threshold diagnostics, and go/no-go gates
selected_candidate_dir = CANDIDATE_DIR / f"train_{SELECTED_TRAIN_SIZE}"
selected_model = tf.keras.models.load_model(
    selected_candidate_dir / "autoencoder.keras", compile=False
)
selected_package = joblib.load(
    selected_candidate_dir / "preprocessing_and_scoring.joblib"
)
validation_auto_score = score_dataframe(selected_model, validation_df, selected_package)

def load_behavior_scores(sample_df):
    parts = []
    for b in BUCKET_IDS:
        keys = sample_df.loc[
            sample_df.d_bucket_id == b, ["transaction_key"]
        ].copy()
        if keys.empty:
            continue
        con.register("sample_keys_current", keys)
        part = con.execute(f"""
            SELECT k.transaction_key, b.behavioral_score
            FROM sample_keys_current k
            LEFT JOIN read_parquet({sql_lit(behavioral_file(b))}) b USING(transaction_key)
        """).df()
        con.unregister("sample_keys_current")
        parts.append(part)
    joined = pd.concat(parts, ignore_index=True)
    if len(joined) != len(sample_df) or joined.transaction_key.duplicated().any():
        raise RuntimeError("Behavioral sample join row/key gate failed")
    score_map = joined.set_index("transaction_key").behavioral_score
    scores = sample_df.transaction_key.map(score_map).to_numpy(dtype="float64")
    if not np.isfinite(scores).all():
        raise RuntimeError("Behavioral sample join contains missing/invalid scores")
    return scores

def union_report_for_split(split, y, auto_score, behavior_score):
    rows = []
    positives = int(y.sum())
    for pct in [0.01, 0.05, 0.10]:
        auto_mask = auto_score >= np.quantile(auto_score, 1 - pct)
        behavior_mask = behavior_score >= np.quantile(behavior_score, 1 - pct)
        union_mask = auto_mask | behavior_mask
        intersection_mask = auto_mask & behavior_mask
        union_rate = float(union_mask.mean())
        equal_budget_behavior = behavior_score >= np.quantile(behavior_score, 1 - union_rate)
        auto_hits = int(y[auto_mask].sum())
        behavior_hits = int(y[behavior_mask].sum())
        union_hits = int(y[union_mask].sum())
        equal_budget_hits = int(y[equal_budget_behavior].sum())
        rows.append({
            "split": split,
            "each_model_top_pct": pct,
            "total_positives": positives,
            "autoencoder_alerts": int(auto_mask.sum()),
            "behavioral_alerts": int(behavior_mask.sum()),
            "union_alerts": int(union_mask.sum()),
            "actual_union_alert_rate": union_rate,
            "autoencoder_positive_hits": auto_hits,
            "behavioral_positive_hits": behavior_hits,
            "intersection_positive_hits": int(y[intersection_mask].sum()),
            "union_positive_hits": union_hits,
            "incremental_autoencoder_hits_over_behavioral": union_hits - behavior_hits,
            "incremental_behavioral_hits_over_autoencoder": union_hits - auto_hits,
            "union_precision": union_hits / union_mask.sum(),
            "union_recall": union_hits / positives,
            "behavioral_recall_at_same_alert_rate": equal_budget_hits / positives,
            "union_minus_equal_budget_behavioral": (union_hits - equal_budget_hits) / positives,
        })
    return pd.DataFrame(rows)

validation_behavior_score = load_behavior_scores(validation_df)
validation_comparison = pd.DataFrame([
    metrics_row("validation", "autoencoder_final64_candidate", y_validation, validation_auto_score),
    metrics_row("validation", "behavioral", y_validation, validation_behavior_score),
])
validation_union = union_report_for_split(
    "validation", y_validation, validation_auto_score, validation_behavior_score
)

threshold_rows = []
for percentile in ALERT_PERCENTILES:
    threshold = float(np.percentile(validation_auto_score, percentile))
    pred = validation_auto_score >= threshold
    threshold_rows.append({
        "percentile": percentile,
        "threshold": threshold,
        "alerts": int(pred.sum()),
        "alert_rate": float(pred.mean()),
        "precision": precision_score(y_validation, pred, zero_division=0),
        "recall": recall_score(y_validation, pred, zero_division=0),
        "f1": f1_score(y_validation, pred, zero_division=0),
    })
validation_thresholds = pd.DataFrame(threshold_rows)
diagnostic_best = validation_thresholds.sort_values(
    ["f1", "precision", "recall"], ascending=False
).iloc[0]
DIAGNOSTIC_THRESHOLD = float(diagnostic_best.threshold)
DIAGNOSTIC_THRESHOLD_PERCENTILE = float(diagnostic_best.percentile)

selected_val = validation_comparison[
    validation_comparison.model == "autoencoder_final64_candidate"
].iloc[0]
top5_union_gain = float(validation_union.loc[
    validation_union.each_model_top_pct == 0.05,
    "union_minus_equal_budget_behavioral",
].iloc[0])

VALIDATION_MODEL_GATE_PASSED = bool(
    selected_val.roc_auc >= MIN_ROC_AUC
    and selected_val.pr_lift >= MIN_PR_LIFT
    and selected_val.recall_top_5pct >= MIN_TOP5_RECALL
)
VALIDATION_COMPLEMENTARITY_PASSED = bool(top5_union_gain > 0)
VALIDATION_GATE_PASSED = bool(
    VALIDATION_MODEL_GATE_PASSED and VALIDATION_COMPLEMENTARITY_PASSED
)

validation_comparison.to_csv(REPORT_DIR / "validation_model_comparison.csv", index=False)
validation_union.to_csv(REPORT_DIR / "validation_union_complementarity.csv", index=False)
validation_thresholds.to_csv(REPORT_DIR / "validation_threshold_diagnostics.csv", index=False)

display(validation_comparison)
display(validation_union)
display(validation_thresholds)
print("Validation model gate:", VALIDATION_MODEL_GATE_PASSED)
print("Validation complementarity gate:", VALIDATION_COMPLEMENTARITY_PASSED)
print("OVERALL VALIDATION GATE:", VALIDATION_GATE_PASSED)
print("Diagnostic-only threshold:", DIAGNOSTIC_THRESHOLD)


### Check whether more Train data improves Validation ranking

Extend the nested sample and compare repeated seeds. The drop at 1.2 million stops the extension and retains 600,000 rows.

In [ ]:
# 11A. Adaptive nested Train expansion and repeated-seed learning curve
ADAPTIVE_TRAIN_SIZES = [600_000, 1_200_000, 2_400_000, 4_800_000]
ADAPTIVE_SEEDS = [42, 123]
PLATEAU_MIN_RELATIVE_PR_AUC_GAIN = 0.05
PLATEAU_MIN_TOP5_RECALL_GAIN = 0.02
MATERIAL_PR_AUC_DROP = -0.05
MATERIAL_TOP5_RECALL_DROP = -0.02
REFERENCE_SAMPLE_SIZE = 600_000

ADAPTIVE_DIR = RUN_DIR / "adaptive_learning_curve"
ADAPTIVE_DIR.mkdir(parents=True, exist_ok=True)


def extend_nested_allocation(allocation, previous_col, target, output_col):
    """Grow an existing nested allocation using remaining stratum capacity."""
    out = allocation.copy().reset_index(drop=True)
    remaining = out.population_rows.astype("int64") - out[previous_col].astype("int64")
    extra_target = int(target - out[previous_col].sum())
    if extra_target <= 0 or extra_target > int(remaining.sum()):
        raise ValueError(f"Invalid extension target {target:,} from {previous_col}")
    exact = remaining / remaining.sum() * extra_target
    extra = np.floor(exact).astype("int64")
    remainder = int(extra_target - extra.sum())
    fractions = exact - np.floor(exact)
    candidates = out.index[extra < remaining]
    order = fractions.loc[candidates].sort_values(ascending=False).index
    extra.loc[order[:remainder]] += 1
    out[output_col] = out[previous_col].astype("int64") + extra
    if int(out[output_col].sum()) != int(target):
        raise AssertionError(f"{output_col} does not sum to {target:,}")
    if (out[output_col] < out[previous_col]).any():
        raise AssertionError(f"{output_col} is not nested above {previous_col}")
    if (out[output_col] > out.population_rows).any():
        raise AssertionError(f"{output_col} exceeds a stratum population")
    return out


adaptive_alloc = train_alloc.copy()
previous_col = "quota_600k"
for size in ADAPTIVE_TRAIN_SIZES[1:]:
    new_col = f"quota_{size // 1000}k"
    adaptive_alloc = extend_nested_allocation(
        adaptive_alloc, previous_col, size, new_col
    )
    previous_col = new_col

adaptive_alloc.to_csv(
    REPORT_DIR / "adaptive_nested_train_allocations.csv", index=False
)


def load_train_increment(previous_size, target_size):
    """Load only new feature rows needed to grow one nested Train sample."""
    previous_col = f"quota_{previous_size // 1000}k"
    target_col = f"quota_{target_size // 1000}k"
    parts = []
    for b in BUCKET_IDS:
        q = adaptive_alloc.loc[
            adaptive_alloc.bucket_id == b,
            ["week_start", previous_col, target_col],
        ].copy()
        con.register("adaptive_quota_current", q)
        part = con.execute(f"""
            WITH base AS (
                SELECT
                    file_row_number,
                    CAST(date_trunc('week', {qid('Timestamp')}) AS DATE) AS week_start,
                    {select_features_sql}
                FROM read_parquet({sql_lit(d_file(b))}, file_row_number=true)
                WHERE {qid('Timestamp')} >= TIMESTAMP {sql_lit(TRAIN_START)}
                  AND {qid('Timestamp')} <  TIMESTAMP {sql_lit(TRAIN_END)}
            ), ranked AS (
                SELECT base.*, q.{qid(previous_col)}, q.{qid(target_col)},
                       ROW_NUMBER() OVER (
                           PARTITION BY base.week_start
                           ORDER BY hash(file_row_number + {SEED + 1000}), file_row_number
                       ) AS sample_rank
                FROM base
                JOIN adaptive_quota_current q USING (week_start)
            )
            SELECT {select_features_sql}
            FROM ranked
            WHERE sample_rank > {qid(previous_col)}
              AND sample_rank <= {qid(target_col)}
        """).df()
        con.unregister("adaptive_quota_current")
        parts.append(part)
        if (b + 1) % 8 == 0:
            print(
                f"Loaded {target_size - previous_size:,} extension rows: "
                f"{b + 1}/64 buckets scanned"
            )
    out = pd.concat(parts, ignore_index=True)[MODEL_RAW_FEATURES]
    expected = int(target_size - previous_size)
    if len(out) != expected:
        raise RuntimeError(
            f"Adaptive extension expected {expected:,} rows, found {len(out):,}"
        )
    return out


#### Model and scoring helpers

In [ ]:


def build_autoencoder_for_seed(input_dim, model_seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(int(model_seed))
    inputs = tf.keras.Input(shape=(input_dim,), name="active_model_features")
    x = tf.keras.layers.GaussianNoise(0.01, name="small_training_noise")(inputs)
    x = tf.keras.layers.Dense(24, activation="relu", name="encoder_24")(x)
    x = tf.keras.layers.Dense(12, activation="relu", name="encoder_12")(x)
    x = tf.keras.layers.Dense(6, activation="relu", name="latent_6")(x)
    x = tf.keras.layers.Dense(12, activation="relu", name="decoder_12")(x)
    x = tf.keras.layers.Dense(24, activation="relu", name="decoder_24")(x)
    outputs = tf.keras.layers.Dense(
        input_dim, activation="linear", name="reconstruction"
    )(x)
    model = tf.keras.Model(
        inputs, outputs, name=f"FINDETECT_Autoencoder_seed_{model_seed}"
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
        loss="mse",
    )
    return model


def reference_matrix(X_train, seed=SEED):
    if len(X_train) <= REFERENCE_SAMPLE_SIZE:
        return X_train
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(len(X_train), size=REFERENCE_SAMPLE_SIZE, replace=False)
    return X_train[idx]


def train_repeated_seeds(train_features, train_size, seeds):
    """Fit one Train-only preprocessor, then train/reuse the requested seeds."""
    size_dir = ADAPTIVE_DIR / f"train_{train_size}"
    size_dir.mkdir(parents=True, exist_ok=True)

    package_base, X_train, _, _, _ = fit_preprocessor(
        train_features, active_features=ACTIVE_MODEL_FEATURES
    )
    X_validation = transform_with_package(validation_df, package_base)
    X_reference = reference_matrix(X_train)
    rows = []

    for model_seed in seeds:
        seed_dir = size_dir / f"seed_{model_seed}"
        seed_dir.mkdir(parents=True, exist_ok=True)
        model_path = seed_dir / "autoencoder.keras"
        package_path = seed_dir / "preprocessing_and_scoring.joblib"
        metrics_path = seed_dir / "validation_metrics.json"

        if (
            not FORCE_REBUILD_CANDIDATES
            and model_path.exists()
            and package_path.exists()
            and metrics_path.exists()
        ):
            row = json.loads(metrics_path.read_text())
            rows.append(row)
            print(f"Train {train_size:,}, seed {model_seed}: reused")
            continue

        model = build_autoencoder_for_seed(X_train.shape[1], model_seed)
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
            ),
        ]
        history = model.fit(
            X_train,
            X_train,
            validation_data=(X_validation, X_validation),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            shuffle=True,
            callbacks=callbacks,
            verbose=1,
        )
        pd.DataFrame(history.history).to_csv(
            seed_dir / "training_history.csv", index=False
        )

        feature_refs, transaction_ref = build_score_references(model, X_reference)
        package = dict(package_base)
        package.update({
            "original_raw_features": ORIGINAL_RAW_FEATURES,
            "model_raw_features": MODEL_RAW_FEATURES,
            "continuous_features": CONTINUOUS_FEATURES,
            "preclean_model_features": PRE_CLEAN_MODEL_FEATURES,
            "feature_error_references": feature_refs,
            "transaction_score_reference": transaction_ref,
            "selected_aggregation_method": SELECTED_AGGREGATION_METHOD,
            "train_sample_size": int(train_size),
            "reference_sample_size": int(len(X_reference)),
            "seed": int(model_seed),
        })
        validation_score = score_dataframe(model, validation_df, package)
        row = metrics_row(
            "validation",
            f"autoencoder_train_{train_size}_seed_{model_seed}",
            y_validation,
            validation_score,
        )
        row.update({
            "train_rows": int(train_size),
            "seed": int(model_seed),
            "epochs_completed": int(len(history.history["loss"])),
            "best_validation_reconstruction_loss": float(
                min(history.history["val_loss"])
            ),
            "model_path": str(model_path),
            "package_path": str(package_path),
        })
        model.save(model_path)
        joblib.dump(package, package_path)
        metrics_path.write_text(json.dumps(row, indent=2))
        rows.append(row)
        print(f"Train {train_size:,}, seed {model_seed}: saved")
        del model, package, validation_score, feature_refs, transaction_ref, history
        gc.collect()

    del package_base, X_train, X_validation, X_reference
    gc.collect()
    return rows


def summarize_repeats(repeat_rows):
    frame = pd.DataFrame(repeat_rows).sort_values(["train_rows", "seed"])
    summary = frame.groupby("train_rows", as_index=False).agg(
        seeds=("seed", "nunique"),
        pr_auc_mean=("pr_auc_average_precision", "mean"),
        pr_auc_std=("pr_auc_average_precision", "std"),
        top5_recall_mean=("recall_top_5pct", "mean"),
        top5_recall_std=("recall_top_5pct", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        reconstruction_loss_mean=("best_validation_reconstruction_loss", "mean"),
    )
    summary["relative_pr_auc_gain"] = summary.pr_auc_mean.pct_change()
    summary["top5_recall_gain"] = summary.top5_recall_mean.diff()
    summary["reconstruction_loss_change"] = (
        summary.reconstruction_loss_mean.diff()
    )
    return frame, summary




#### Compare and apply the stopping rule

In [ ]:
# Existing seed-42 600K result is reused; only the missing repeat is trained.
baseline = learning_curve.loc[learning_curve.train_rows == 600_000].iloc[0].to_dict()
baseline.update({
    "seed": int(SEED),
    "model_path": str(CANDIDATE_DIR / "train_600000" / "autoencoder.keras"),
    "package_path": str(
        CANDIDATE_DIR / "train_600000" / "preprocessing_and_scoring.joblib"
    ),
})
adaptive_repeat_rows = [baseline]
adaptive_train_features = train_master_df[MODEL_RAW_FEATURES].copy()
adaptive_repeat_rows.extend(
    train_repeated_seeds(adaptive_train_features, 600_000, [123])
)

ADAPTIVE_STOP_REASON = None
ADAPTIVE_CURVE_READY_FOR_TEST = False
ADAPTIVE_SELECTED_TRAIN_SIZE = None
previous_size = 600_000

for target_size in ADAPTIVE_TRAIN_SIZES[1:]:
    increment = load_train_increment(previous_size, target_size)
    adaptive_train_features = pd.concat(
        [adaptive_train_features, increment], ignore_index=True
    )
    del increment
    gc.collect()
    if len(adaptive_train_features) != target_size:
        raise RuntimeError(
            f"Expected {target_size:,} adaptive Train rows, "
            f"found {len(adaptive_train_features):,}"
        )

    adaptive_repeat_rows.extend(
        train_repeated_seeds(adaptive_train_features, target_size, ADAPTIVE_SEEDS)
    )
    adaptive_repeats, adaptive_summary = summarize_repeats(adaptive_repeat_rows)
    current = adaptive_summary.loc[
        adaptive_summary.train_rows == target_size
    ].iloc[0]
    relative_ap_gain = float(current.relative_pr_auc_gain)
    top5_gain = float(current.top5_recall_gain)

    material_decline = bool(
        relative_ap_gain <= MATERIAL_PR_AUC_DROP
        or top5_gain <= MATERIAL_TOP5_RECALL_DROP
    )
    plateau = bool(
        relative_ap_gain < PLATEAU_MIN_RELATIVE_PR_AUC_GAIN
        and top5_gain < PLATEAU_MIN_TOP5_RECALL_GAIN
    )

    display(adaptive_summary)
    print(
        f"{previous_size:,} -> {target_size:,}: "
        f"mean relative PR-AUC gain={relative_ap_gain:.2%}, "
        f"mean Top-5 recall gain={top5_gain:.2%}"
    )

    if material_decline:
        ADAPTIVE_STOP_REASON = "material_decline"
        ADAPTIVE_SELECTED_TRAIN_SIZE = previous_size
        ADAPTIVE_CURVE_READY_FOR_TEST = True
        break
    if plateau:
        ADAPTIVE_STOP_REASON = "plateau"
        ADAPTIVE_SELECTED_TRAIN_SIZE = previous_size
        ADAPTIVE_CURVE_READY_FOR_TEST = True
        break
    previous_size = target_size

if ADAPTIVE_STOP_REASON is None:
    ADAPTIVE_STOP_REASON = "maximum_checked_without_plateau"
    ADAPTIVE_SELECTED_TRAIN_SIZE = ADAPTIVE_TRAIN_SIZES[-1]
    ADAPTIVE_CURVE_READY_FOR_TEST = False

adaptive_repeats, adaptive_summary = summarize_repeats(adaptive_repeat_rows)
adaptive_repeats.to_csv(
    REPORT_DIR / "adaptive_learning_curve_repeats.csv", index=False
)
adaptive_summary.to_csv(
    REPORT_DIR / "adaptive_learning_curve_summary.csv", index=False
)

display(adaptive_repeats[[
    "train_rows", "seed", "pr_auc_average_precision", "recall_top_5pct",
    "roc_auc", "best_validation_reconstruction_loss"
]])
display(adaptive_summary)
print("Adaptive stop reason:", ADAPTIVE_STOP_REASON)
print("Adaptive selected Train size:", f"{ADAPTIVE_SELECTED_TRAIN_SIZE:,}")
print("Test may be opened:", ADAPTIVE_CURVE_READY_FOR_TEST)


## Freeze the winner and confirm on Test

Reload the selected model and fitted transformations, record Validation metrics, and evaluate the frozen choice once on Test.

In [ ]:
# 11B. Freeze the adaptive winner and rebuild Validation gates before Test
selected_repeat = adaptive_repeats.loc[
    adaptive_repeats.train_rows == ADAPTIVE_SELECTED_TRAIN_SIZE
].sort_values(
    ["pr_auc_average_precision", "recall_top_5pct", "roc_auc"],
    ascending=False,
).iloc[0]

SELECTED_TRAIN_SIZE = int(selected_repeat.train_rows)
SELECTED_MODEL_SEED = int(selected_repeat.seed)
selected_model = tf.keras.models.load_model(
    Path(selected_repeat.model_path), compile=False
)
selected_package = joblib.load(Path(selected_repeat.package_path))
validation_auto_score = score_dataframe(
    selected_model, validation_df, selected_package
)

validation_comparison = pd.DataFrame([
    metrics_row(
        "validation",
        f"autoencoder_adaptive_{SELECTED_TRAIN_SIZE}_seed_{SELECTED_MODEL_SEED}",
        y_validation,
        validation_auto_score,
    ),
    metrics_row(
        "validation", "behavioral", y_validation, validation_behavior_score
    ),
])
validation_union = union_report_for_split(
    "validation", y_validation, validation_auto_score, validation_behavior_score
)

threshold_rows = []
for percentile in ALERT_PERCENTILES:
    threshold = float(np.percentile(validation_auto_score, percentile))
    pred = validation_auto_score >= threshold
    threshold_rows.append({
        "percentile": percentile,
        "threshold": threshold,
        "alerts": int(pred.sum()),
        "alert_rate": float(pred.mean()),
        "precision": precision_score(y_validation, pred, zero_division=0),
        "recall": recall_score(y_validation, pred, zero_division=0),
        "f1": f1_score(y_validation, pred, zero_division=0),
    })
validation_thresholds = pd.DataFrame(threshold_rows)
diagnostic_best = validation_thresholds.sort_values(
    ["f1", "precision", "recall"], ascending=False
).iloc[0]
DIAGNOSTIC_THRESHOLD = float(diagnostic_best.threshold)
DIAGNOSTIC_THRESHOLD_PERCENTILE = float(diagnostic_best.percentile)

selected_val = validation_comparison.iloc[0]
top5_union_gain = float(validation_union.loc[
    validation_union.each_model_top_pct == 0.05,
    "union_minus_equal_budget_behavioral",
].iloc[0])
VALIDATION_MODEL_GATE_PASSED = bool(
    selected_val.roc_auc >= MIN_ROC_AUC
    and selected_val.pr_lift >= MIN_PR_LIFT
    and selected_val.recall_top_5pct >= MIN_TOP5_RECALL
)
VALIDATION_COMPLEMENTARITY_PASSED = bool(top5_union_gain > 0)
VALIDATION_GATE_PASSED = bool(
    VALIDATION_MODEL_GATE_PASSED and VALIDATION_COMPLEMENTARITY_PASSED
)

adaptive_decision = {
    "selected_train_size": SELECTED_TRAIN_SIZE,
    "selected_seed": SELECTED_MODEL_SEED,
    "stop_reason": ADAPTIVE_STOP_REASON,
    "test_may_be_opened": ADAPTIVE_CURVE_READY_FOR_TEST,
    "plateau_rule": {
        "relative_pr_auc_gain_below": PLATEAU_MIN_RELATIVE_PR_AUC_GAIN,
        "top5_recall_gain_below": PLATEAU_MIN_TOP5_RECALL_GAIN,
        "both_required": True,
    },
    "material_decline_rule": {
        "relative_pr_auc_gain_at_most": MATERIAL_PR_AUC_DROP,
        "top5_recall_gain_at_most": MATERIAL_TOP5_RECALL_DROP,
        "either_sufficient": True,
    },
    "selection_split": "validation",
    "test_was_loaded_before_selection": False,
}
(REPORT_DIR / "adaptive_train_size_decision.json").write_text(
    json.dumps(adaptive_decision, indent=2)
)
validation_comparison.to_csv(
    REPORT_DIR / "validation_model_comparison_adaptive.csv", index=False
)
validation_union.to_csv(
    REPORT_DIR / "validation_union_complementarity_adaptive.csv", index=False
)
validation_thresholds.to_csv(
    REPORT_DIR / "validation_threshold_diagnostics_adaptive.csv", index=False
)

display(validation_comparison)
display(validation_union)
display(validation_thresholds)
print(json.dumps(adaptive_decision, indent=2))
print("Validation model gate:", VALIDATION_MODEL_GATE_PASSED)
print("Validation complementarity gate:", VALIDATION_COMPLEMENTARITY_PASSED)
print("OVERALL VALIDATION GATE:", VALIDATION_GATE_PASSED)

if not ADAPTIVE_CURVE_READY_FOR_TEST:
    print(
        "STOP: no plateau was found through 4.8M. Do not open Test; "
        "extend the curve beyond 4.8M first."
    )
elif VALIDATION_GATE_PASSED:
    print("PASS: Train size and seed are frozen. The Test checkpoint may run once.")
else:
    print("STOP: adaptive size is frozen, but the Validation gate failed.")

del adaptive_train_features
gc.collect()


In [ ]:
# 12. Open Test exactly once after all choices are frozen
if not globals().get("ADAPTIVE_CURVE_READY_FOR_TEST", False):
    raise RuntimeError(
        "STOP: adaptive learning curve has not frozen the Train size. Test remains unopened."
    )
if not VALIDATION_GATE_PASSED:
    raise RuntimeError("STOP: Validation gate failed. Test remains unopened.")

test_df = load_fixed_split_sample("test", test_alloc, 3000)
y_test = test_df.evaluation_label.astype("int8").to_numpy()
test_auto_score = score_dataframe(selected_model, test_df, selected_package)
test_behavior_score = load_behavior_scores(test_df)

test_comparison = pd.DataFrame([
    metrics_row("test", "autoencoder_final64_frozen", y_test, test_auto_score),
    metrics_row("test", "behavioral", y_test, test_behavior_score),
])
test_union = union_report_for_split("test", y_test, test_auto_score, test_behavior_score)

test_pred = test_auto_score >= DIAGNOSTIC_THRESHOLD
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
test_threshold = pd.DataFrame([{
    "threshold_source": "validation_diagnostic_only",
    "selected_percentile": DIAGNOSTIC_THRESHOLD_PERCENTILE,
    "threshold": DIAGNOSTIC_THRESHOLD,
    "alerts": int(test_pred.sum()),
    "alert_rate": float(test_pred.mean()),
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "f1": f1_score(y_test, test_pred, zero_division=0),
    "tn": int(cm[0, 0]), "fp": int(cm[0, 1]),
    "fn": int(cm[1, 0]), "tp": int(cm[1, 1]),
}])

selected_test = test_comparison[
    test_comparison.model == "autoencoder_final64_frozen"
].iloc[0]
test_top5_union_gain = float(test_union.loc[
    test_union.each_model_top_pct == 0.05,
    "union_minus_equal_budget_behavioral",
].iloc[0])
TEST_CONFIRMATION_PASSED = bool(
    selected_test.roc_auc >= MIN_ROC_AUC
    and selected_test.pr_lift >= MIN_PR_LIFT
    and selected_test.recall_top_5pct >= MIN_TOP5_RECALL
    and test_top5_union_gain > 0
)

test_comparison.to_csv(REPORT_DIR / "test_model_comparison_frozen.csv", index=False)
test_union.to_csv(REPORT_DIR / "test_union_complementarity_frozen.csv", index=False)
test_threshold.to_csv(REPORT_DIR / "test_diagnostic_threshold_frozen.csv", index=False)

display(test_comparison)
display(test_union)
display(test_threshold)
print("TEST CONFIRMATION PASSED:", TEST_CONFIRMATION_PASSED)


## Save the final training artifacts

Save the model, preprocessing and score references, and decision reports for subsequent full-population scoring.

In [ ]:
# 13. Save the complete Final64 training package and decision record
ELIGIBLE_FOR_FULL_POPULATION_CANARY = bool(
    VALIDATION_GATE_PASSED and TEST_CONFIRMATION_PASSED
)

final_package = dict(selected_package)
final_package.update({
    "original_raw_features": ORIGINAL_RAW_FEATURES,
    "model_raw_features": MODEL_RAW_FEATURES,
    "continuous_features": CONTINUOUS_FEATURES,
    "binary_source_features": BINARY_SOURCE_FEATURES,
    "derived_binary_features": DERIVED_BINARY_FEATURES,
    "preclean_model_features": PRE_CLEAN_MODEL_FEATURES,
    "active_model_features": ACTIVE_MODEL_FEATURES,
    "near_zero_features_from_600k_schema": near_zero_features,
    "correlation_drops_from_600k_schema": correlation_drops,
    "selected_aggregation_method": SELECTED_AGGREGATION_METHOD,
    "selected_train_sample_size": SELECTED_TRAIN_SIZE,
    "selected_model_seed": SELECTED_MODEL_SEED,
    "diagnostic_validation_threshold": DIAGNOSTIC_THRESHOLD,
    "diagnostic_validation_threshold_percentile": DIAGNOSTIC_THRESHOLD_PERCENTILE,
    "operational_threshold_status": "not approved; choose only after multi-model fusion",
    "validation_gate_passed": VALIDATION_GATE_PASSED,
    "test_confirmation_passed": TEST_CONFIRMATION_PASSED,
    "eligible_for_full_population_canary": ELIGIBLE_FOR_FULL_POPULATION_CANARY,
    "temporal_contract": {
        "train": [TRAIN_START, TRAIN_END],
        "validation": [TRAIN_END, VAL_END],
        "test": [VAL_END, TEST_END],
    },
    "modeling_population_rows": EXPECTED_MODELING_ROWS,
    "transaction_key_mode": TRANSACTION_KEY_MODE,
    "transaction_key": (
        "existing D transaction_key"
        if HAS_SOURCE_TRANSACTION_KEY
        else "B{bucket_id:03d}_R{file_row_number:012d}"
    ),
    "label_policy": "Is Laundering used for evaluation only; Pattern Type never loaded",
})

selected_model.save(FINAL_MODEL_DIR / "autoencoder_final64.keras")
joblib.dump(
    final_package,
    FINAL_MODEL_DIR / "preprocessing_and_scoring_final64.joblib",
)

metadata = {
    "project": "FINDETECT",
    "artifact": "autoencoder_final64_training",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_signature": signature,
    "bucket_ids": BUCKET_IDS,
    "selected_train_sample_size": SELECTED_TRAIN_SIZE,
    "selected_model_seed": SELECTED_MODEL_SEED,
    "validation_rows": len(validation_df),
    "test_rows": len(test_df),
    "selected_aggregation_method": SELECTED_AGGREGATION_METHOD,
    "transaction_score_reference_saved": True,
    "validation_gate_passed": VALIDATION_GATE_PASSED,
    "test_confirmation_passed": TEST_CONFIRMATION_PASSED,
    "eligible_for_full_population_canary": ELIGIBLE_FOR_FULL_POPULATION_CANARY,
    "next_step": (
        "score full bucket_000 with the frozen package"
        if ELIGIBLE_FOR_FULL_POPULATION_CANARY
        else "stop; do not score the full population"
    ),
}
(FINAL_MODEL_DIR / "metadata_final64.json").write_text(json.dumps(metadata, indent=2))
(REPORT_DIR / "final64_decision.json").write_text(json.dumps(metadata, indent=2))

print(json.dumps(metadata, indent=2))
print("Final package:", FINAL_MODEL_DIR)
print("Reports:", REPORT_DIR)
if ELIGIBLE_FOR_FULL_POPULATION_CANARY:
    print("PASS: next step is one full-bucket canary on bucket_000, not all 64 yet.")
else:
    print("STOP: do not start full-population scoring.")
